In [45]:
# Data Ingestion

In [46]:
import os
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "config" / "config.yaml").exists() and (candidate / "src").exists():
            return candidate
        for child in candidate.iterdir():
            if child.is_dir() and (child / "config" / "config.yaml").exists() and (child / "src").exists():
                return child
    raise FileNotFoundError("Could not find project root containing config/config.yaml")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT


PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [47]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

    


In [48]:
from src.Text_Summarizer.utils.common import read_yaml, create_directories
from src.Text_Summarizer.constants import *

In [49]:
class ConfigurationManager:
    def __init__(self, config_filepath: Path = CONFIG_FILE_PATH, params_filepath: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [50]:
import os
import urllib.request as request
import zipfile
from src.Text_Summarizer.logger import logger


In [51]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=str(self.config.source_URL),
                filename=str(self.config.local_data_file)
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {os.path.getsize(self.config.local_data_file) / (1024 * 1024)} MB")

    def unzip_and_clean(self):
        unzip_dir = self.config.unzip_dir
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_dir)

In [52]:
config = ConfigurationManager()
data_ingestion_config= config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_data()
data_ingestion.unzip_and_clean()



[2026-07-04 22:27:43,418: INFO: common]: YAML file: config/config.yaml loaded successfully.
[2026-07-04 22:27:43,419: INFO: common]: YAML file: params.yaml loaded successfully.
[2026-07-04 22:27:43,420: INFO: common]: Directory created at: artifacts
[2026-07-04 22:27:43,421: INFO: common]: Directory created at: artifacts/data_ingestion
[2026-07-04 22:27:46,213: INFO: 2927394386]: artifacts/data_ingestion/summarizer.zip downloaded! with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 12D2:67E16:2C0C:7F60:6A48FC3E
Accept-Ranges: bytes
Date: Sat, 04 Jul 2026 12:27:44 GMT
Via: 1.1 varnish
X-Served-By: cache-bne-ybbn1320032-B